In [1]:
import ray
from ray import serve
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import pyarrow as pa
import requests
from starlette.requests import Request
import json
from sentence_transformers import SentenceTransformer
import textdistance
import numpy as np

# Model Inference for Batch and Online Use Cases with Ray Data and Ray Serve

## Batch inference

Typical task: for each user in a dataset of users, find products to recommend, generate an email with those recommendations, and write out to storage/mail queue/DB

## Online inference

Typical tasks:

* Given a live user session, generate a recommendation
* Given a live user session and a search query, generate a product search result *and* a product recommendation

We will implement these use cases

We'll use an existing recommender model snapshot for these activities

In [2]:
! aws s3 sync s3://anyscale-public-materials-use2/ecom /mnt/cluster_storage/ecom

download: s3://anyscale-public-materials-use2/ecom/catalog/27_00db830ac2654c5f8012007929524983_000001_000000-0.parquet to ../../../mnt/cluster_storage/ecom/catalog/27_00db830ac2654c5f8012007929524983_000001_000000-0.parquet
download: s3://anyscale-public-materials-use2/ecom/catalog/27_00db830ac2654c5f8012007929524983_000000_000000-0.parquet to ../../../mnt/cluster_storage/ecom/catalog/27_00db830ac2654c5f8012007929524983_000000_000000-0.parquet
download: s3://anyscale-public-materials-use2/ecom/cat_with_embeddings/57_52effbfc1db2476c9c27d26ae7a15c12_000003_000000-0.parquet to ../../../mnt/cluster_storage/ecom/cat_with_embeddings/57_52effbfc1db2476c9c27d26ae7a15c12_000003_000000-0.parquet
download: s3://anyscale-public-materials-use2/ecom/catalog/27_00db830ac2654c5f8012007929524983_000002_000000-0.parquet to ../../../mnt/cluster_storage/ecom/catalog/27_00db830ac2654c5f8012007929524983_000002_000000-0.parquet
download: s3://anyscale-public-materials-use2/ecom/cat_with_embeddings/57_52effb

In [3]:
base_model_path = '/mnt/cluster_storage/ecom/recommender/base_model/model.pt'

## Batch inference

Batch inference is typically implemented with Ray Data pipelines

Here is our synthetic user dataset

In [4]:
! head -29 /mnt/cluster_storage/ecom/users.json

[
  {
    "id": "779e23ec-4714-4ba3-bc3a-2c8487cde669",
    "first_name": "Lucas",
    "last_name": "Rodriguez",
    "email": "lucas.rodriguez8648@hotmail.com",
    "last_20_positive_item_interactions": [
      624,
      208,
      730,
      714,
      897,
      86,
      964,
      267,
      574,
      260,
      880,
      600,
      645,
      882,
      135,
      273,
      657,
      203,
      123,
      974
    ]
  },


In [5]:
ds = ray.data.read_json('/mnt/cluster_storage/ecom/users.json')

2026-08-24 14:44:28,469	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 100.125.138.24:6379...
2026-08-24 14:44:28,496	INFO worker.py:2003 -- Connected to Ray cluster. View the dashboard at https://session-iy1mz6uapeim3bm6iel8iqvdpm.i.anyscaleuserdata.com 
2026-08-24 14:44:28,499	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_d78b25acd631f0c560f6e293bce17943053dc5fb.zip' (0.50MiB) to Ray cluster...
2026-08-24 14:44:28,502	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_d78b25acd631f0c560f6e293bce17943053dc5fb.zip'.
/home/ray/anaconda3/lib/python3.11/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


This dataset is small, so we can materialize it to inspect and test

In [6]:
ds.materialize()

2026-08-24 14:44:30,772	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-08-24 14:44:30,844	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:44:30,844	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles]
2026-08-24 14:44:30,848	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 28.0% of available memory (26.9GiB out of 96.0GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
2026-08-24 14:44:30,851	INFO __init__.py:56 -- Progress will be logged because stdout is 

shape: (1000, 5)
╭──────────────────────────────────────┬────────────┬───────────┬─────────────────────────────────┬──────────────────────────────────────────╮
│ id                                   ┆ first_name ┆ last_name ┆ email                           ┆ last_20_positive_item_interactions       │
│ ---                                  ┆ ---        ┆ ---       ┆ ---                             ┆ ---                                      │
│ string                               ┆ string     ┆ string    ┆ string                          ┆ list<item: int64>                        │
╞══════════════════════════════════════╪════════════╪═══════════╪═════════════════════════════════╪══════════════════════════════════════════╡
│ 779e23ec-4714-4ba3-bc3a-2c8487cde669 ┆ Lucas      ┆ Rodriguez ┆ lucas.rodriguez8648@hotmail.com ┆ [624, 208, 730, 714, 897, 86, 964, 267,… │
│ 10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745 ┆ Emily      ┆ Taylor    ┆ emily.taylor7287@icloud.com     ┆ [6, 612, 536, 705, 647, 7

Ok, but note the warning in the logs: PyArrow is not successfully parsing our JSON

In [7]:
try:
    pa.json.read_json('/mnt/cluster_storage/ecom/users.json')
except Exception as e:
    print(e)
    

JSON parse error: Column() changed from object to array in row 0


Maybe we should use JSON lines:

In [8]:
! head /mnt/cluster_storage/ecom/users.ndjson

{"id": "779e23ec-4714-4ba3-bc3a-2c8487cde669", "first_name": "Lucas", "last_name": "Rodriguez", "email": "lucas.rodriguez8648@hotmail.com", "last_20_positive_item_interactions": [624, 208, 730, 714, 897, 86, 964, 267, 574, 260, 880, 600, 645, 882, 135, 273, 657, 203, 123, 974]}
{"id": "10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745", "first_name": "Emily", "last_name": "Taylor", "email": "emily.taylor7287@icloud.com", "last_20_positive_item_interactions": [6, 612, 536, 705, 647, 704, 615, 73, 930, 350, 68, 324, 48, 446, 373, 160, 129, 418, 846, 318]}
{"id": "7034dd99-ceb3-474d-a0ba-5beaf122273f", "first_name": "William", "last_name": "Martinez", "email": "william.martinez7222@yahoo.com", "last_20_positive_item_interactions": [74, 875, 87, 859, 271, 184, 704, 616, 812, 547, 719, 549, 624, 26, 255, 889, 994, 326, 809, 653]}
{"id": "8beef414-1c38-4ee1-821f-46613c1b0503", "first_name": "Alexander", "last_name": "Lee", "email": "alexander.lee8701@outlook.com", "last_20_positive_item_interactions": [5

In [9]:
ds = ray.data.read_json('/mnt/cluster_storage/ecom/users.ndjson', lines=True)
ds.materialize()

/home/ray/anaconda3/lib/python3.11/site-packages/ray/anyscale/data/api/read_api.py:586: UserWarning: orjson provides the fastest `read_json` implementation, but it’s not installed. Falling back to pandas. To use orjson, run: `pip install orjson`.
  warnings.warn(
2026-08-24 14:44:42,981	INFO logging.py:416 -- Registered dataset logger for dataset dataset_4_0
2026-08-24 14:44:42,986	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_4_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:44:42,986	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_4_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles]
2026-08-24 14:44:43,007	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_4_0 =======
2026-08-24 14:44:43,008	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 14:44:43,008	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CP

MaterializedDataset(num_rows=0, schema=Unknown schema)

No error or warning ... but no data!

How do we troubleshoot? Let's see if PyArrow reads this data properly.

In [10]:
pa.json.read_json('/mnt/cluster_storage/ecom/users.ndjson')

pyarrow.Table
id: string
first_name: string
last_name: string
email: string
last_20_positive_item_interactions: list<item: int64>
  child 0, item: int64
----
id: [["779e23ec-4714-4ba3-bc3a-2c8487cde669","10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745","7034dd99-ceb3-474d-a0ba-5beaf122273f","8beef414-1c38-4ee1-821f-46613c1b0503","62445079-71af-4a46-95b6-1706e6ce5832",...,"b81bd6e2-0f72-43d5-91bd-3019a6e573c0","f720bee7-48bf-41f6-be48-cfe3aba3fa39","cf7733df-e2db-4212-8003-69d37bd25dae","3239c639-b6f0-4d6d-b918-5b98f493eb5f","a1fa562b-f8a2-401a-8ef4-a39164aa3279"]]
first_name: [["Lucas","Emily","William","Alexander","James",...,"Evelyn","William","Liam","Emma","William"]]
last_name: [["Rodriguez","Taylor","Martinez","Lee","Lee",...,"Martin","Brown","Moore","Lee","Lopez"]]
email: [["lucas.rodriguez8648@hotmail.com","emily.taylor7287@icloud.com","william.martinez7222@yahoo.com","alexander.lee8701@outlook.com","james.lee1268@gmail.com",...,"evelyn.martin3670@icloud.com","william.brown6105@icloud.com"

In [11]:
pd.read_json('/mnt/cluster_storage/ecom/users.ndjson', lines=True)

,id,first_name,last_name,email,last_20_positive_item_interactions
0,779e23ec-4714-4ba3-bc3a-2c8487cde669,Lucas,Rodriguez,lucas.rodriguez8648@hotmail.com,"[624, 208, 730, 714, 897, 86, 964, 267, 574, 2..."
1,10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745,Emily,Taylor,emily.taylor7287@icloud.com,"[6, 612, 536, 705, 647, 704, 615, 73, 930, 350..."
2,7034dd99-ceb3-474d-a0ba-5beaf122273f,William,Martinez,william.martinez7222@yahoo.com,"[74, 875, 87, 859, 271, 184, 704, 616, 812, 54..."
3,8beef414-1c38-4ee1-821f-46613c1b0503,Alexander,Lee,alexander.lee8701@outlook.com,"[519, 139, 195, 589, 933, 58, 799, 798, 500, 3..."
4,62445079-71af-4a46-95b6-1706e6ce5832,James,Lee,james.lee1268@gmail.com,"[750, 329, 244, 483, 831, 158, 352, 180, 416, ..."
...,...,...,...,...,...
995,b81bd6e2-0f72-43d5-91bd-3019a6e573c0,Evelyn,Martin,evelyn.martin3670@icloud.com,"[790, 335, 531, 588, 349, 808, 456, 379, 995, ..."
996,f720bee7-48bf-41f6-be48-cfe3aba3fa39,William,Brown,william.brown6105@icloud.com,"[214, 100, 702, 793, 136, 699, 404, 584, 561, ..."
997,cf7733df-e2db-4212-8003-69d37bd25dae,Liam,Moore,liam.moore812@outlook.com,"[574, 134, 642, 912, 861, 629, 919, 140, 350, ..."
998,3239c639-b6f0-4d6d-b918-5b98f493eb5f,Emma,Lee,emma.lee2300@yahoo.com,"[586, 157, 559, 346, 654, 236, 566, 573, 184, ..."


PyArrow and Pandas are happy, so the data and libs are ok. Ray Data is filtering out non `.json` files by default

In [12]:
ds = ray.data.read_json('/mnt/cluster_storage/ecom/users.ndjson', lines=True, file_extensions=['.ndjson'])
ds.materialize()

/home/ray/anaconda3/lib/python3.11/site-packages/ray/anyscale/data/api/read_api.py:586: UserWarning: orjson provides the fastest `read_json` implementation, but it’s not installed. Falling back to pandas. To use orjson, run: `pip install orjson`.
  warnings.warn(
2026-08-24 14:44:43,588	INFO logging.py:416 -- Registered dataset logger for dataset dataset_7_0
2026-08-24 14:44:43,592	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_7_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:44:43,592	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_7_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles]
2026-08-24 14:44:43,609	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_7_0 =======
2026-08-24 14:44:43,610	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 14:44:43,610	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CP

shape: (1000, 5)
╭──────────────────────────────────────┬────────────┬───────────┬─────────────────────────────────┬──────────────────────────────────────────╮
│ id                                   ┆ first_name ┆ last_name ┆ email                           ┆ last_20_positive_item_interactions       │
│ ---                                  ┆ ---        ┆ ---       ┆ ---                             ┆ ---                                      │
│ string                               ┆ string     ┆ string    ┆ string                          ┆ object                                   │
╞══════════════════════════════════════╪════════════╪═══════════╪═════════════════════════════════╪══════════════════════════════════════════╡
│ 779e23ec-4714-4ba3-bc3a-2c8487cde669 ┆ Lucas      ┆ Rodriguez ┆ lucas.rodriguez8648@hotmail.com ┆ [624, 208, 730, 714, 897, 86, 964, 267,… │
│ 10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745 ┆ Emily      ┆ Taylor    ┆ emily.taylor7287@icloud.com     ┆ [6, 612, 536, 705, 647, 7

Given our data and our recommendation model, the plan is to factor our logic into composable parts for a pipeline:

1. read users
1. convert user IDs (GUID) to index ints for the recommender
1. infer product recommendation indexes
1. map product indexes to product records and put relevant details into the dataset
1. generate email for each record in the pipeline
1. save or enqueue (for mailing) each email

Our real-world systems will likely include databases for looking up users and products, so let's simulate a database and its API.

This example encapsulates the queries.

How will various parts of our Ray code find this database facade? We can implement it as a named Ray Actor and then look it up as needed.

In [13]:
@ray.remote
class DatabaseFacade():
    def __init__(self, users, products):
        self.users = pd.read_json(users, lines=True)
        self.products = pd.read_parquet(products)
        
    def users_for_ids(self, ids):
        return self.users[self.users['id'].isin(ids)]
    
    def products_for_indices(self, idxs):
        return self.products.iloc[idxs]

DatabaseFacade.options(name="database").remote('/mnt/cluster_storage/ecom/users.ndjson', '/mnt/cluster_storage/ecom/cat_with_embeddings')

Actor(DatabaseFacade, 40ee33d6a2e37036d38c3ea302000000)

In [14]:
my_db = ray.get_actor("database")

ref = my_db.users_for_ids.remote(['7034dd99-ceb3-474d-a0ba-5beaf122273f', 'cf7733df-e2db-4212-8003-69d37bd25dae'])

When calling remote methods (tasks) in a Ray program, we get `ObjectRef`s, a form of distributed pointer and promise.

When we (or Ray itself) need the referenced data, Ray can retrieve that data over the wire and deserialize to a Python object.

In [15]:
ray.get(ref)

,id,first_name,last_name,email,last_20_positive_item_interactions
2,7034dd99-ceb3-474d-a0ba-5beaf122273f,William,Martinez,william.martinez7222@yahoo.com,"[74, 875, 87, 859, 271, 184, 704, 616, 812, 54..."
997,cf7733df-e2db-4212-8003-69d37bd25dae,Liam,Moore,liam.moore812@outlook.com,"[574, 134, 642, 912, 861, 629, 919, 140, 350, ..."


In [16]:
ray.get(ref).index.values

array([  2, 997])

Our conversion from user records to indices can be done with `map_batches` and a stateless function (since the state is handled by the database)

In [17]:
def get_user_indices(batch):
    my_db = ray.get_actor("database")
    ref = my_db.users_for_ids.remote(batch['id'])
    batch['user_index'] = ray.get(ref).index.values
    return batch

In [18]:
ds.map_batches(get_user_indices).take_batch(4)

2026-08-24 14:44:45,974	INFO logging.py:416 -- Registered dataset logger for dataset dataset_10_0
2026-08-24 14:44:45,980	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_10_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:44:45,981	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_10_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> TaskPoolMapOperator[MapBatches(get_user_indices)] -> LimitOperator[limit=4]
2026-08-24 14:44:46,004	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_10_0 =======
2026-08-24 14:44:46,005	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 14:44:46,006	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store
2026-08-24 14:44:46,007	INFO logging_progress.py:181 -- 
2026-08-24 14:44:46,007	INFO logging_progress.py:231 -- ListFiles: 0/1
2026-08-24 14:44:46,008	I

{'id': array(['779e23ec-4714-4ba3-bc3a-2c8487cde669',
        '10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745',
        '7034dd99-ceb3-474d-a0ba-5beaf122273f',
        '8beef414-1c38-4ee1-821f-46613c1b0503'], dtype=object),
 'first_name': array(['Lucas', 'Emily', 'William', 'Alexander'], dtype=object),
 'last_name': array(['Rodriguez', 'Taylor', 'Martinez', 'Lee'], dtype=object),
 'email': array(['lucas.rodriguez8648@hotmail.com', 'emily.taylor7287@icloud.com',
        'william.martinez7222@yahoo.com', 'alexander.lee8701@outlook.com'],
       dtype=object),
 'last_20_positive_item_interactions': array([array([624, 208, 730, 714, 897,  86, 964, 267, 574, 260, 880, 600, 645,
               882, 135, 273, 657, 203, 123, 974])                             ,
        array([  6, 612, 536, 705, 647, 704, 615,  73, 930, 350,  68, 324,  48,
               446, 373, 160, 129, 418, 846, 318])                             ,
        array([ 74, 875,  87, 859, 271, 184, 704, 616, 812, 547, 719, 549, 624,
      

Next, we'll load and use our recommender. In production, we'd import this but we can define it inline here to see the full code

In [19]:
# minimal_two_tower.py
class TwoTower(nn.Module):
    def __init__(self, num_users: int, num_items: int, dim: int = 64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)

        # optional: small projection MLPs (kept minimal)
        self.user_proj = nn.Identity()
        self.item_proj = nn.Identity()

    def encode_users(self, user_ids: torch.LongTensor) -> torch.Tensor:
        u = self.user_proj(self.user_emb(user_ids))
        return F.normalize(u, dim=-1)

    def encode_items(self, item_ids: torch.LongTensor) -> torch.Tensor:
        v = self.item_proj(self.item_emb(item_ids))
        return F.normalize(v, dim=-1)

    def forward(self, user_ids: torch.LongTensor, pos_item_ids: torch.LongTensor):
        """
        Returns logits matrix [B,B] where diagonal is the positive pair and
        off-diagonals are in-batch negatives.
        """
        u = self.encode_users(user_ids)         # [B, D]
        v = self.encode_items(pos_item_ids)     # [B, D]
        logits = u @ v.t()                      # [B, B]
        return logits

Recall the batch inference pattern with Ray Data: `map_batches` using a stateful Actor class, where the Actor loads the model in its constructor.

Recall also that this pattern does not require `@ray.remote`: Ray will convert this class to an Actor for us.

In [20]:
class Recommend():
    def __init__(self, model_location, num_recommendations, num_users, num_items):
        self.model = TwoTower(num_users, num_items)
        self.model.load_state_dict(torch.load(model_location, weights_only=True))
        self.model.eval()
        self.num_recommendations = num_recommendations
        self.num_items = num_items
        
    @torch.no_grad()
    def recommend_topk_batch(self,
        user_ids: torch.LongTensor,     # [B]
        all_item_ids: torch.LongTensor, # [N]
    ):
        """
        Returns:
          topk_indices: [B, k]  (item ids)
          topk_scores:  [B, k]
        """
        model = self.model
        k = self.num_recommendations
        model.eval()

        # Encode
        u = model.encode_users(user_ids)     # [B, D]
        v = model.encode_items(all_item_ids) # [N, D]

        # Similarity
        scores = u @ v.t()                   # [B, N]

        # Top-k per user
        topk_scores, topk_idx = torch.topk(scores, k=k, dim=1)

        # Map indices back to item ids
        topk_item_ids = all_item_ids[topk_idx]  # [B, k]

        return topk_item_ids, topk_scores
    
    def recommend_for_users(self, users):
        (items, scores) = self.recommend_topk_batch(torch.tensor(users), torch.arange(1, self.num_items))
        return items.numpy()
        
    def __call__(self, batch, column):
        users = batch[column]
        batch['recommended_items'] = self.recommend_for_users(users)
        return batch

In [21]:
sample_batch = ds \
    .map_batches(get_user_indices) \
    .map_batches(Recommend, fn_constructor_args=[base_model_path, 3, 1000, 1000], fn_args=['user_index']).take_batch(4)

sample_batch

2026-08-24 14:44:46,752	INFO logging.py:416 -- Registered dataset logger for dataset dataset_13_0
2026-08-24 14:44:46,760	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_13_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:44:46,760	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_13_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[MapBatches(get_user_indices)->MapBatches(Recommend)] -> LimitOperator[limit=4]
{"asctime":"2026-08-24 14:44:46,802","levelname":"E","message":"Actor with class name: 'MapWorker(MapBatches(get_user_indices)->MapBatches(Recommend))' and ID: '8f6af5a0c4bb29727191e83e02000000' has constructor arguments in the object store and max_restarts > 0. If the arguments in the object store go out of scope or are lost, the actor restart will fail. See https://github.com/ray-project/ray/issues/53727 for more details."

{'id': array(['779e23ec-4714-4ba3-bc3a-2c8487cde669',
        '10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745',
        '7034dd99-ceb3-474d-a0ba-5beaf122273f',
        '8beef414-1c38-4ee1-821f-46613c1b0503'], dtype=object),
 'first_name': array(['Lucas', 'Emily', 'William', 'Alexander'], dtype=object),
 'last_name': array(['Rodriguez', 'Taylor', 'Martinez', 'Lee'], dtype=object),
 'email': array(['lucas.rodriguez8648@hotmail.com', 'emily.taylor7287@icloud.com',
        'william.martinez7222@yahoo.com', 'alexander.lee8701@outlook.com'],
       dtype=object),
 'last_20_positive_item_interactions': array([array([624, 208, 730, 714, 897,  86, 964, 267, 574, 260, 880, 600, 645,
               882, 135, 273, 657, 203, 123, 974])                             ,
        array([  6, 612, 536, 705, 647, 704, 615,  73, 930, 350,  68, 324,  48,
               446, 373, 160, 129, 418, 846, 318])                             ,
        array([ 74, 875,  87, 859, 271, 184, 704, 616, 812, 547, 719, 549, 624,
      

In [22]:
sample_batch['recommended_items']

array([[164, 268, 219],
       [780, 504, 659],
       [395, 487, 751],
       [247, 827, 649]])

Let's build up code to get some item details (name, description, and price) for each recommended item. We'll need those for the emails.

This is an interesting task because it probably makes sense to get all of the recommendations for a batch of users (i.e., num_recos_per_user * num_users) all at once from our model. 

We can use the sample batch above to develop/test/debug our code line by line before wiring it up to the Ray Data pipeline

In [23]:
products_ref = ray.get_actor("database").products_for_indices.remote(sample_batch['recommended_items'].flatten())
products = ray.get(products_ref)
products

,mfg_item_id,parent_id,item_id,cat,name,desc,price,desc_emb
164,GLD-GBY5U16SY,PARENT-F0MBALY,VEN-ECOKNPOCG,Office & Stationery,Notebook (Hardcover) (Heavyweight) (White),"A sturdy hardcover notebook for notes, lists, ...",43.00,"[-0.13644713, 0.03422939, -0.0035965603, 0.014..."
268,SUM-FIHXY0S26,,MKP-VNHV0RNNA,Garden & Outdoor Living,"Outdoor Doormat (Compact) (Size S, Pink)",A textured doormat that helps trap dirt before...,123.99,"[-0.14090228, 0.0049562952, -0.003176659, 0.02..."
219,ECL-LI1RQPTQH,PARENT-L30CN5V,SHO-5P8HZKU3G,Grocery & Pantry,Organic Granola (Family Pack) (1 lb),Crunchy granola with oats and nuts for quick b...,26.99,"[-0.105562605, -0.035462625, -0.00932526, -0.0..."
780,PRM-4N85H6NFP,,VEN-O0EMY4KUN,Housewares,Glass Meal-Prep Container (Dishwasher-Safe) (B...,A leak-resistant glass container designed for ...,29.99,"[-0.16319618, 0.022852635, 0.025880972, 0.0168..."
504,PRM-1659K9I2I,PARENT-1SL0MDF,MKP-2M8EQAOGA,Apparel,"Men's Classic Chino Pants (Size S, Beige)","Straight-fit chinos with a versatile, office-t...",52.99,"[-0.1085049, -0.008239723, 0.043322317, 0.0282..."
659,RIV-JV5I09KHV,PARENT-WHOAF7P,VEN-0I8L4XGVW,Tools,Claw Hammer - Quick-Release (18V),A balanced hammer for driving nails and pullin...,146.49,"[-0.08570439, 0.015190384, 0.021112284, 0.0108..."
395,PRM-UGQDYGJUM,PARENT-9ZWQ9SU,SHO-C70NS1L7I,Housewares,LED Under-Cabinet Light Bar (Slim Profile) (Red),A slim LED light bar to brighten counters and ...,13.00,"[-0.16433638, 0.0014552865, 0.0354154, 0.02025..."
487,ALP-L0OKO74MX,PARENT-VZYQTEE,CAT-X8Y6OKF0V,Grocery & Pantry,Rice (Long Grain) (Low Sugar) (32 oz),Long grain rice that cooks fluffy for bowls an...,5.99,"[-0.08890704, -0.014625957, -0.008743354, -0.0..."
751,SUM-86T19PBED,PARENT-PBIN8C3,VEN-489ZNGGOE,Electronics,"Wireless Earbuds - Extended Battery (64GB, Teal)",Compact wireless earbuds with a stable fit for...,92.00,"[-0.095001616, -0.006837907, 0.0026027963, -0...."
247,DUR-V38CBYENY,,MKP-7T10TLGTE,Books,Kids' Science Experiments (Illustrated Edition...,Hands-on experiments that make science fun at ...,39.49,"[-0.1279442, 0.01138331, -0.00066488114, 0.002..."


In [24]:
products.iloc[0:3]

,mfg_item_id,parent_id,item_id,cat,name,desc,price,desc_emb
164,GLD-GBY5U16SY,PARENT-F0MBALY,VEN-ECOKNPOCG,Office & Stationery,Notebook (Hardcover) (Heavyweight) (White),"A sturdy hardcover notebook for notes, lists, ...",43.00,"[-0.13644713, 0.03422939, -0.0035965603, 0.014..."
268,SUM-FIHXY0S26,,MKP-VNHV0RNNA,Garden & Outdoor Living,"Outdoor Doormat (Compact) (Size S, Pink)",A textured doormat that helps trap dirt before...,123.99,"[-0.14090228, 0.0049562952, -0.003176659, 0.02..."
219,ECL-LI1RQPTQH,PARENT-L30CN5V,SHO-5P8HZKU3G,Grocery & Pantry,Organic Granola (Family Pack) (1 lb),Crunchy granola with oats and nuts for quick b...,26.99,"[-0.105562605, -0.035462625, -0.00932526, -0.0..."


In [25]:
products.iloc[0:3][['name', 'desc', 'price']]

,name,desc,price
164,Notebook (Hardcover) (Heavyweight) (White),"A sturdy hardcover notebook for notes, lists, ...",43.00
268,"Outdoor Doormat (Compact) (Size S, Pink)",A textured doormat that helps trap dirt before...,123.99
219,Organic Granola (Family Pack) (1 lb),Crunchy granola with oats and nuts for quick b...,26.99


In [26]:
my_db = ray.get_actor("database")
users_in_batch = len(sample_batch['id'])
recs_per_user = sample_batch['recommended_items'].shape[1] # recos will be shape (users in batch, recos per user)
product_indices = sample_batch['recommended_items'].flatten()
ref = my_db.products_for_indices.remote(product_indices)
recommendations = ray.get(ref)
recommendations    

,mfg_item_id,parent_id,item_id,cat,name,desc,price,desc_emb
164,GLD-GBY5U16SY,PARENT-F0MBALY,VEN-ECOKNPOCG,Office & Stationery,Notebook (Hardcover) (Heavyweight) (White),"A sturdy hardcover notebook for notes, lists, ...",43.00,"[-0.13644713, 0.03422939, -0.0035965603, 0.014..."
268,SUM-FIHXY0S26,,MKP-VNHV0RNNA,Garden & Outdoor Living,"Outdoor Doormat (Compact) (Size S, Pink)",A textured doormat that helps trap dirt before...,123.99,"[-0.14090228, 0.0049562952, -0.003176659, 0.02..."
219,ECL-LI1RQPTQH,PARENT-L30CN5V,SHO-5P8HZKU3G,Grocery & Pantry,Organic Granola (Family Pack) (1 lb),Crunchy granola with oats and nuts for quick b...,26.99,"[-0.105562605, -0.035462625, -0.00932526, -0.0..."
780,PRM-4N85H6NFP,,VEN-O0EMY4KUN,Housewares,Glass Meal-Prep Container (Dishwasher-Safe) (B...,A leak-resistant glass container designed for ...,29.99,"[-0.16319618, 0.022852635, 0.025880972, 0.0168..."
504,PRM-1659K9I2I,PARENT-1SL0MDF,MKP-2M8EQAOGA,Apparel,"Men's Classic Chino Pants (Size S, Beige)","Straight-fit chinos with a versatile, office-t...",52.99,"[-0.1085049, -0.008239723, 0.043322317, 0.0282..."
659,RIV-JV5I09KHV,PARENT-WHOAF7P,VEN-0I8L4XGVW,Tools,Claw Hammer - Quick-Release (18V),A balanced hammer for driving nails and pullin...,146.49,"[-0.08570439, 0.015190384, 0.021112284, 0.0108..."
395,PRM-UGQDYGJUM,PARENT-9ZWQ9SU,SHO-C70NS1L7I,Housewares,LED Under-Cabinet Light Bar (Slim Profile) (Red),A slim LED light bar to brighten counters and ...,13.00,"[-0.16433638, 0.0014552865, 0.0354154, 0.02025..."
487,ALP-L0OKO74MX,PARENT-VZYQTEE,CAT-X8Y6OKF0V,Grocery & Pantry,Rice (Long Grain) (Low Sugar) (32 oz),Long grain rice that cooks fluffy for bowls an...,5.99,"[-0.08890704, -0.014625957, -0.008743354, -0.0..."
751,SUM-86T19PBED,PARENT-PBIN8C3,VEN-489ZNGGOE,Electronics,"Wireless Earbuds - Extended Battery (64GB, Teal)",Compact wireless earbuds with a stable fit for...,92.00,"[-0.095001616, -0.006837907, 0.0026027963, -0...."
247,DUR-V38CBYENY,,MKP-7T10TLGTE,Books,Kids' Science Experiments (Illustrated Edition...,Hands-on experiments that make science fun at ...,39.49,"[-0.1279442, 0.01138331, -0.00066488114, 0.002..."


Next we have to split these recommendations up to match them to the respective users

In [27]:
[recommendations[['name', 'desc', 'price']].iloc[i*recs_per_user:(i+1)*recs_per_user] for i in range(users_in_batch)]

[                                           name  \
 164  Notebook (Hardcover) (Heavyweight) (White)   
 268    Outdoor Doormat (Compact) (Size S, Pink)   
 219        Organic Granola (Family Pack) (1 lb)   
 
                                                   desc   price  
 164  A sturdy hardcover notebook for notes, lists, ...   43.00  
 268  A textured doormat that helps trap dirt before...  123.99  
 219  Crunchy granola with oats and nuts for quick b...   26.99  ,
                                                   name  \
 780  Glass Meal-Prep Container (Dishwasher-Safe) (B...   
 504          Men's Classic Chino Pants (Size S, Beige)   
 659                  Claw Hammer - Quick-Release (18V)   
 
                                                   desc   price  
 780  A leak-resistant glass container designed for ...   29.99  
 504  Straight-fit chinos with a versatile, office-t...   52.99  
 659  A balanced hammer for driving nails and pullin...  146.49  ,
                      

And wrap that logic in a function

In [28]:
def get_recommended_items_details(batch):
    my_db = ray.get_actor("database")
    users_in_batch = len(batch['id'])
    recs_per_user = batch['recommended_items'].shape[1] # recos will be shape (users in batch, recos per user)
    
    product_indices = batch['recommended_items'].flatten()
    ref = my_db.products_for_indices.remote(product_indices)
    recommendations = ray.get(ref)
    
    recs_split_by_users = [recommendations[['name', 'desc', 'price']].iloc[i*recs_per_user : (i+1)*recs_per_user] for i in range(users_in_batch)]
    batch['recommended_items_details'] = recs_split_by_users
    return batch

... for use with `map_batches`

In [29]:
samples = ds \
    .map_batches(get_user_indices) \
    .map_batches(Recommend, fn_constructor_args=[base_model_path, 3, 1000, 1000], fn_args=['user_index']) \
    .map_batches(get_recommended_items_details) \
    .take(2)

samples

2026-08-24 14:44:52,922	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-08-24 14:44:52,925	INFO logging.py:416 -- Registered dataset logger for dataset dataset_17_0
2026-08-24 14:44:52,930	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_17_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:44:52,931	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_17_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[MapBatches(get_user_indices)->MapBatches(Recommend)] -> TaskPoolMapOperator[MapBatches(get_recommended_items_details)] -> LimitOperator[limit=2]
2026-08-24 14:44:53,090	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_17_0 =======
2026-08-24 14:44:53,090	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 14:44:53,091	INFO log

[{'id': '779e23ec-4714-4ba3-bc3a-2c8487cde669',
  'first_name': 'Lucas',
  'last_name': 'Rodriguez',
  'email': 'lucas.rodriguez8648@hotmail.com',
  'last_20_positive_item_interactions': array([624, 208, 730, 714, 897,  86, 964, 267, 574, 260, 880, 600, 645,
         882, 135, 273, 657, 203, 123, 974]),
  'user_index': 0,
  'recommended_items': array([164, 268, 219]),
  'recommended_items_details':                                            name  \
  164  Notebook (Hardcover) (Heavyweight) (White)   
  268    Outdoor Doormat (Compact) (Size S, Pink)   
  219        Organic Granola (Family Pack) (1 lb)   
  
                                                    desc   price  
  164  A sturdy hardcover notebook for notes, lists, ...   43.00  
  268  A textured doormat that helps trap dirt before...  123.99  
  219  Crunchy granola with oats and nuts for quick b...   26.99  },
 {'id': '10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745',
  'first_name': 'Emily',
  'last_name': 'Taylor',
  'email': 'emily

In [30]:
samples[0]['recommended_items_details'].iloc[0]['name']

'Notebook (Hardcover) (Heavyweight) (White)'

Note the warning in the logs: the pandas DataFrame we're using for our data records is not serializing in the optimal way into a PyArrow array.

We could refactor this code to use alternate record representations if this proves to be a performance issue.

The code for generating emails doesn't trivially vectorize, so we'll start with a per-record (Ray Dataset `map`) approach.

In [31]:
def generate_email_for_user(user):
    templated_mail = f'''Greetings, {user['first_name']} {user['last_name']}

    Check out the following personalized recommendations!

    {'; '.join([user['recommended_items_details'].iloc[i]['desc'] 
    + ' Only ' 
    + str(user['recommended_items_details'].iloc[i]['price']) for i in range(len(user['recommended_items_details']))])}

    Click here to unsubscribe.
    '''
    
    user['reco_email'] = templated_mail
    
    return user

In [32]:
generate_email_for_user(samples[0])

{'id': '779e23ec-4714-4ba3-bc3a-2c8487cde669',
 'first_name': 'Lucas',
 'last_name': 'Rodriguez',
 'email': 'lucas.rodriguez8648@hotmail.com',
 'last_20_positive_item_interactions': array([624, 208, 730, 714, 897,  86, 964, 267, 574, 260, 880, 600, 645,
        882, 135, 273, 657, 203, 123, 974]),
 'user_index': 0,
 'recommended_items': array([164, 268, 219]),
 'recommended_items_details':                                            name  \
 164  Notebook (Hardcover) (Heavyweight) (White)   
 268    Outdoor Doormat (Compact) (Size S, Pink)   
 219        Organic Granola (Family Pack) (1 lb)   
 
                                                   desc   price  
 164  A sturdy hardcover notebook for notes, lists, ...   43.00  
 268  A textured doormat that helps trap dirt before...  123.99  
 219  Crunchy granola with oats and nuts for quick b...   26.99  ,
 'reco_email': 'Greetings, Lucas Rodriguez\n\n    Check out the following personalized recommendations!\n\n    A sturdy hardcover not

In [33]:
sample_batch = ds \
    .map_batches(get_user_indices) \
    .map_batches(Recommend, fn_constructor_args=[base_model_path, 3, 1000, 1000], fn_args=['user_index']) \
    .map_batches(get_recommended_items_details) \
    .map(generate_email_for_user) \
    .take_batch(2)

sample_batch

2026-08-24 14:44:58,798	INFO logging.py:416 -- Registered dataset logger for dataset dataset_22_0
2026-08-24 14:44:58,804	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_22_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:44:58,805	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_22_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[MapBatches(get_user_indices)->MapBatches(Recommend)] -> TaskPoolMapOperator[MapBatches(get_recommended_items_details)] -> LimitOperator[limit=2] -> TaskPoolMapOperator[Map(generate_email_for_user)]
2026-08-24 14:44:58,965	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_22_0 =======
2026-08-24 14:44:58,966	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 14:44:58,967	INFO logging_progress.py:227 -- Active & requested resources: 0/16 CPU, 0.0B/9.0GiB object store (pending: 

{'id': array(['779e23ec-4714-4ba3-bc3a-2c8487cde669',
        '10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745'], dtype=object),
 'first_name': array(['Lucas', 'Emily'], dtype=object),
 'last_name': array(['Rodriguez', 'Taylor'], dtype=object),
 'email': array(['lucas.rodriguez8648@hotmail.com', 'emily.taylor7287@icloud.com'],
       dtype=object),
 'last_20_positive_item_interactions': array([[624, 208, 730, 714, 897,  86, 964, 267, 574, 260, 880, 600, 645,
         882, 135, 273, 657, 203, 123, 974],
        [  6, 612, 536, 705, 647, 704, 615,  73, 930, 350,  68, 324,  48,
         446, 373, 160, 129, 418, 846, 318]]),
 'user_index': array([0, 1]),
 'recommended_items': array([[164, 268, 219],
        [780, 504, 659]]),
 'recommended_items_details': array([                                           name  \
        164  Notebook (Hardcover) (Heavyweight) (White)
        268    Outdoor Doormat (Compact) (Size S, Pink)
        219        Organic Granola (Family Pack) (1 lb)
 
                     

We'll simulate an adapter for an email sending queue. We'll use `map_batches` to give it batches of user emails to send. The sending will be a side effect, the "output" will be a column in our dataset indicating which emails have been queued, and we'll log some queue info as a side effect as well.

In [34]:
class EmailSendingQueue:
    def __init__(self, email_queue_service_info):
        self.queue = { 
            'service' : email_queue_service_info,
            'out_queue' : [] 
        }
    
    def __call__(self, batch):
        for email in batch['reco_email']:
            self.queue['out_queue'].append(email)
        batch['queued'] = [True] * len(batch['reco_email'])
        print(f'Currently queued {len(self.queue["out_queue"])} messages.')
        return batch

(Map(generate_email_for_user) pid=29305, ip=100.113.73.39) Failed to convert column 'recommended_items_details' into pyarrow array due to: Error converting data to Arrow: column: 'recommended_items_details', shape: (2, 3, 3), dtype: object, data: [[['Notebook (Hardcover) (Heavyweight) (White)'
(Map(generate_email_for_user) pid=29305, ip=100.113.73.39)    'A sturdy hardcover notebook for notes, lists, and sketches. Helps keep y...; falling back to serialize as pickled python objects
(Map(generate_email_for_user) pid=29305, ip=100.113.73.39) Traceback (most recent call last):
(Map(generate_email_for_user) pid=29305, ip=100.113.73.39)   File "/home/ray/anaconda3/lib/python3.11/site-packages/ray/data/_internal/tensor_extensions/arrow.py", line 875, in from_numpy
(Map(generate_email_for_user) pid=29305, ip=100.113.73.39)     return cls._from_numpy(arr)
(Map(generate_email_for_user) pid=29305, ip=100.113.73.39)            ^^^^^^^^^^^^^^^^^^^^
(Map(generate_email_for_user) pid=29305, ip=100.1

In [35]:
sample_batch = ds \
    .map_batches(get_user_indices) \
    .map_batches(Recommend, fn_constructor_args=[base_model_path, 3, 1000, 1000], fn_args=['user_index']) \
    .map_batches(get_recommended_items_details) \
    .map(generate_email_for_user) \
    .map_batches(EmailSendingQueue, batch_size=32, fn_constructor_args=['sender:bulk:12.34.5678']) \
    .take_batch(2)

sample_batch

2026-08-24 14:45:04,203	INFO logging.py:416 -- Registered dataset logger for dataset dataset_28_0
2026-08-24 14:45:04,208	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_28_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:45:04,208	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_28_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[MapBatches(get_user_indices)->MapBatches(Recommend)] -> ActorPoolMapOperator[MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)] -> LimitOperator[limit=2]
2026-08-24 14:45:04,262	WARNING utils.py:33 -- Truncating long operator name to 100 characters. To disable this behavior, set `ray.data.DataContext.get_current().DEFAULT_ENABLE_PROGRESS_BAR_NAME_TRUNCATION = False`.
2026-08-24 14:45:04,394	INFO logging_progress.py:174 -- ======= Running Dataset: dat

(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 32 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 64 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 96 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 128 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 160 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip

2026-08-24 14:45:14,438	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_28_0 =======
2026-08-24 14:45:14,439	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 14:45:14,440	INFO logging_progress.py:227 -- Active & requested resources: 1/16 CPU, 289.7KiB/9.0GiB object store
2026-08-24 14:45:14,440	INFO logging_progress.py:181 -- 
2026-08-24 14:45:14,441	INFO logging_progress.py:231 -- ListFiles: 1/1
2026-08-24 14:45:14,441	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-08-24 14:45:14,442	INFO logging_progress.py:231 -- ReadFiles: 1000/1000
2026-08-24 14:45:14,442	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-08-24 14:45:14,443	INFO logging_progress.py:231 -- MapBatches(get_user_indices)->MapBatches(Recommend): 1000/1000
2026-08-24 14:45:14,444	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blo

(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 768 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 800 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 832 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 864 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 896 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570,

2026-08-24 14:45:15,783	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_28_0 execution finished in 11.57 seconds


(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29570, ip=100.113.73.39) Currently queued 1000 messages.


{'id': array(['779e23ec-4714-4ba3-bc3a-2c8487cde669',
        '10fdbf5f-2dcc-4e35-bd1c-a2ce6bdd8745'], dtype=object),
 'first_name': array(['Lucas', 'Emily'], dtype=object),
 'last_name': array(['Rodriguez', 'Taylor'], dtype=object),
 'email': array(['lucas.rodriguez8648@hotmail.com', 'emily.taylor7287@icloud.com'],
       dtype=object),
 'last_20_positive_item_interactions': array([[624, 208, 730, 714, 897,  86, 964, 267, 574, 260, 880, 600, 645,
         882, 135, 273, 657, 203, 123, 974],
        [  6, 612, 536, 705, 647, 704, 615,  73, 930, 350,  68, 324,  48,
         446, 373, 160, 129, 418, 846, 318]]),
 'user_index': array([0, 1]),
 'recommended_items': array([[164, 268, 219],
        [780, 504, 659]]),
 'recommended_items_details': array([                                           name  \
        164  Notebook (Hardcover) (Heavyweight) (White)
        268    Outdoor Doormat (Compact) (Size S, Pink)
        219        Organic Granola (Family Pack) (1 lb)
 
                     

Note that although we were only trying to take a small batch, all 1000 records got processed. Why?

Our dataset is all in one block (because it's so small) and Ray Data must process an entire block.

> Check that if we repartition (to create more blocks), Ray does not process all 1000 records

This is not a big issue for us here, but this could be a big problem if we were iteratively developing code that uses an LLM where each inference takes a lot of time.

Now we're ready to generate all of our marketing emails and queue for sending

In [36]:
ds \
    .map_batches(get_user_indices) \
    .map_batches(Recommend, fn_constructor_args=[base_model_path, 3, 1000, 1000], fn_args=['user_index']) \
    .map_batches(get_recommended_items_details) \
    .map(generate_email_for_user) \
    .map_batches(EmailSendingQueue, batch_size=32, fn_constructor_args=['sender:bulk:12.34.5678']) \
    .count()

2026-08-24 14:45:15,948	INFO logging.py:416 -- Registered dataset logger for dataset dataset_34_0
2026-08-24 14:45:15,954	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_34_0. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
2026-08-24 14:45:15,955	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_34_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> ActorPoolMapOperator[MapBatches(get_user_indices)->MapBatches(Recommend)] -> ActorPoolMapOperator[MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)] -> TaskPoolMapOperator[Project] -> AggregateNumRows[AggregateNumRows]
2026-08-24 14:45:16,164	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_34_0 =======
2026-08-24 14:45:16,165	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 14:45:16,166	INFO logging_progress.py:227 -- Active & requested res

(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip=100.113.73.39) Currently queued 32 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip=100.113.73.39) Currently queued 64 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip=100.113.73.39) Currently queued 96 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip=100.113.73.39) Currently queued 128 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip=100.113.73.39) Currently queued 160 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip

2026-08-24 14:45:26,170	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_34_0 =======
2026-08-24 14:45:26,171	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-08-24 14:45:26,172	INFO logging_progress.py:227 -- Active & requested resources: 1/16 CPU, 289.7KiB/9.0GiB object store
2026-08-24 14:45:26,172	INFO logging_progress.py:181 -- 
2026-08-24 14:45:26,172	INFO logging_progress.py:231 -- ListFiles: 1/1
2026-08-24 14:45:26,173	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-08-24 14:45:26,173	INFO logging_progress.py:231 -- ReadFiles: 1000/1000
2026-08-24 14:45:26,174	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blocks: 0 (0.0B); Resources: 0.0 CPU, 0.0B object store
2026-08-24 14:45:26,174	INFO logging_progress.py:231 -- MapBatches(get_user_indices)->MapBatches(Recommend): 1000/1000
2026-08-24 14:45:26,175	INFO logging_progress.py:233 --   Tasks: 0; Actors: 0; Queued blo

(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip=100.113.73.39) Currently queued 768 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip=100.113.73.39) Currently queued 800 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip=100.113.73.39) Currently queued 832 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip=100.113.73.39) Currently queued 864 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654, ip=100.113.73.39) Currently queued 896 messages.
(MapWorker(MapBatches(get_recommended_items_details)->Map(generate_email_for_user)->MapBatches(EmailSendingQueue)) pid=29654,

2026-08-24 14:45:27,456	INFO streaming_executor.py:294 -- ✔️  Dataset dataset_34_0 execution finished in 11.50 seconds


1000

## Online services: recommendation and search

Using Ray Serve, we'll implement the following use case first: 

* Given a user ID, we'll generate recommendations for them

In [37]:
recommender = Recommend(base_model_path, 3, 1000, 1000)

In [38]:
ref = my_db.users_for_ids.remote(['7034dd99-ceb3-474d-a0ba-5beaf122273f'])

In [39]:
user_df = ray.get(ref)
user_df

,id,first_name,last_name,email,last_20_positive_item_interactions
2,7034dd99-ceb3-474d-a0ba-5beaf122273f,William,Martinez,william.martinez7222@yahoo.com,"[74, 875, 87, 859, 271, 184, 704, 616, 812, 54..."


In [40]:
user_df.index.values

array([2])

In [41]:
recos = recommender.recommend_for_users(user_df.index.values)
recos

array([[395, 487, 751]])

In [42]:
ray.get(my_db.products_for_indices.remote(recos.flatten()))

,mfg_item_id,parent_id,item_id,cat,name,desc,price,desc_emb
395,PRM-UGQDYGJUM,PARENT-9ZWQ9SU,SHO-C70NS1L7I,Housewares,LED Under-Cabinet Light Bar (Slim Profile) (Red),A slim LED light bar to brighten counters and ...,13.00,"[-0.16433638, 0.0014552865, 0.0354154, 0.02025..."
487,ALP-L0OKO74MX,PARENT-VZYQTEE,CAT-X8Y6OKF0V,Grocery & Pantry,Rice (Long Grain) (Low Sugar) (32 oz),Long grain rice that cooks fluffy for bowls an...,5.99,"[-0.08890704, -0.014625957, -0.008743354, -0.0..."
751,SUM-86T19PBED,PARENT-PBIN8C3,VEN-489ZNGGOE,Electronics,"Wireless Earbuds - Extended Battery (64GB, Teal)",Compact wireless earbuds with a stable fit for...,92.00,"[-0.095001616, -0.006837907, 0.0026027963, -0...."


We can start with a single Ray Serve logic component (a Deployment) that handles our request I/O as well as our recommendation code.

Once that works, we'll refactor to create better separation of concerns.

Here's a basic Deployment service that combines the Ingress (I/O handling+routing) role and the Recommender role.

In [43]:
@serve.deployment()
class IngressAndRecommender:
    def __init__(self, base_model_path: str, num_recos: int, num_users: int, num_products: int):
        self.db = ray.get_actor("database")
        self.recommender = Recommend(base_model_path, num_recos, num_users, num_products)

    async def __call__(self, request: Request):  # __call__ takes a Request object
        user = await request.json()
        return self.recommend([user['id']])[['item_id', 'name', 'desc', 'price']].to_json()
    
    def recommend(self, user_ids):
        ref = self.db.users_for_ids.remote(user_ids)
        user_df = ray.get(ref)
        recos = self.recommender.recommend_for_users(user_df.index.values)
        return ray.get(self.db.products_for_indices.remote(recos.flatten()))

In [44]:
bound_deployment = IngressAndRecommender.bind(base_model_path, 3, 1000, 1000)
app_handle = serve.run(bound_deployment)

(ProxyActor pid=62610) INFO 2026-08-24 14:45:32,437 proxy 100.125.138.24 -- Proxy starting on node f987380542f8cbfb6253e55ae77bf5ade2c020d1e77df80460d6d8da (HTTP port: 8000).
INFO 2026-08-24 14:45:32,592 serve 54397 -- Started Serve in namespace "serve".
(ServeController pid=62540) INFO 2026-08-24 14:45:32,673 controller 62540 -- Deploying new version of Deployment(name='IngressAndRecommender', app='default') (initial target replicas: 1).
(ProxyActor pid=62610) INFO 2026-08-24 14:45:32,588 proxy 100.125.138.24 -- Got updated endpoints: {}.
(ProxyActor pid=62610) INFO 2026-08-24 14:45:32,676 proxy 100.125.138.24 -- Got updated endpoints: {Deployment(name='IngressAndRecommender', app='default'): EndpointInfo(route='/', app_is_cross_language=False, route_patterns=None)}.
(ServeController pid=62540) INFO 2026-08-24 14:45:32,776 controller 62540 -- Adding 1 replica to Deployment(name='IngressAndRecommender', app='default').
(ProxyActor pid=62610) INFO 2026-08-24 14:45:32,684 proxy 100.125.1

In [45]:
await app_handle.recommend.remote(['7034dd99-ceb3-474d-a0ba-5beaf122273f'])

(ProxyActor pid=29816, ip=100.113.73.39) INFO 2026-08-24 14:45:37,912 proxy 100.113.73.39 -- Started <ray.serve._private.router.SharedRouterLongPollClient object at 0x702d9a8ddb90>.
INFO 2026-08-24 14:45:38,061 serve 54397 -- Started <ray.serve._private.router.SharedRouterLongPollClient object at 0x7422dc5f9f90>.


,mfg_item_id,parent_id,item_id,cat,name,desc,price,desc_emb
395,PRM-UGQDYGJUM,PARENT-9ZWQ9SU,SHO-C70NS1L7I,Housewares,LED Under-Cabinet Light Bar (Slim Profile) (Red),A slim LED light bar to brighten counters and ...,13.00,"[-0.16433638, 0.0014552865, 0.0354154, 0.02025..."
487,ALP-L0OKO74MX,PARENT-VZYQTEE,CAT-X8Y6OKF0V,Grocery & Pantry,Rice (Long Grain) (Low Sugar) (32 oz),Long grain rice that cooks fluffy for bowls an...,5.99,"[-0.08890704, -0.014625957, -0.008743354, -0.0..."
751,SUM-86T19PBED,PARENT-PBIN8C3,VEN-489ZNGGOE,Electronics,"Wireless Earbuds - Extended Battery (64GB, Teal)",Compact wireless earbuds with a stable fit for...,92.00,"[-0.095001616, -0.006837907, 0.0026027963, -0...."


In [46]:
response = requests.post("http://localhost:8000/", json={ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f"})

response.json()

(ServeReplica:default:IngressAndRecommender pid=29746, ip=100.113.73.39) /home/ray/anaconda3/lib/python3.11/site-packages/ray/serve/_private/replica.py:3153: UserWarning: Calling sync method 'recommend' directly on the asyncio loop. In a future version, sync methods will be run in a threadpool by default. Ensure your sync methods are thread safe or keep the existing behavior by making them `async def`. Opt into the new behavior by setting RAY_SERVE_RUN_SYNC_IN_THREADPOOL=1.
(ServeReplica:default:IngressAndRecommender pid=29746, ip=100.113.73.39)   warnings.warn(
(ServeReplica:default:IngressAndRecommender pid=29746, ip=100.113.73.39) INFO 2026-08-24 14:45:38,094 default_IngressAndRecommender w7hfvrjf 69a3c910-adec-42f8-85f9-8b972d959dd5 -- CALL recommend OK 16.5ms


{'item_id': {'395': 'SHO-C70NS1L7I',
  '487': 'CAT-X8Y6OKF0V',
  '751': 'VEN-489ZNGGOE'},
 'name': {'395': 'LED Under-Cabinet Light Bar (Slim Profile) (Red)',
  '487': 'Rice (Long Grain) (Low Sugar) (32 oz)',
  '751': 'Wireless Earbuds - Extended Battery (64GB, Teal)'},
 'desc': {'395': 'A slim LED light bar to brighten counters and workspaces. Easy to wipe clean and store between uses.',
  '487': 'Long grain rice that cooks fluffy for bowls and sides. Packed for freshness with resealable packaging.',
  '751': 'Compact wireless earbuds with a stable fit for daily listening. Includes a quick-start guide and standard charging support.'},
 'price': {'395': 13.0, '487': 5.99, '751': 92.0}}

Since we'll be substantially re-arranging this app as we develop we can shut down everything as needed by calling `serve.shutdown`

In [47]:
serve.shutdown()

(ServeReplica:default:IngressAndRecommender pid=29746, ip=100.113.73.39) INFO 2026-08-24 14:45:38,331 default_IngressAndRecommender w7hfvrjf 02eb8ce3-e3cf-4846-8aa2-c45182ca5549 -- POST / 200 16.2ms
(ServeController pid=62540) INFO 2026-08-24 14:45:38,551 controller 62540 -- Removing 1 replica from Deployment(name='IngressAndRecommender', app='default').
(ServeController pid=62540) INFO 2026-08-24 14:45:40,570 controller 62540 -- Replica(id='w7hfvrjf', deployment='IngressAndRecommender', app='default') is stopped.


(raylet) Task ServeController.listen_for_change failed. There are infinite retries remaining, so the task will be retried. Error: The actor is dead because it was killed by `ray.kill`.
(raylet) Task ServeController.graceful_shutdown failed. There are infinite retries remaining, so the task will be retried. Error: The actor is dead because it was killed by `ray.kill`.


Let's make a more single-purpose Ingress Deployment

In [48]:
@serve.deployment()
class Ingress:
    def __init__(self, recommender):
        self.recommender = recommender

    async def __call__(self, request: Request):  # __call__ takes a Request object
        user = await request.json()
        return (await self.recommender.recommend.remote([user['id']]))[['item_id', 'name', 'desc', 'price']].to_json()

We'll make the Recommender its own Deployment (component).

To do this, the logic will stay the same but we'll need to make a minor change in how we get remote results.

* We cannot `ray.get` a response from a deployment handle (if you try, you'll get an error that says exactly that).
* Instead, we `await` the response. Similar idea, but matches async pattern for web apps.

Let's also make the DatabaseFacade into a Ray Serve Deployment -- that will make it more robust (e.g., Ray Serve will restart it if it fails) and easier to autoscale.

In [49]:
@serve.deployment()
class Recommender:
    def __init__(self, base_model_path: str, num_recos: int, num_users: int, num_products: int, database):
        self.recommender = Recommend(base_model_path, num_recos, num_users, num_products)
        self.db = database
    
    async def recommend(self, user_ids):
        ref = self.db.users_for_ids.remote(user_ids)
        user_df = await ref
        recos = self.recommender.recommend_for_users(user_df.index.values)
        return await self.db.products_for_indices.remote(recos.flatten())

In [50]:
@serve.deployment()
class DatabaseFacade():
    def __init__(self, users, products):
        self.users = pd.read_json(users, lines=True)
        self.products = pd.read_parquet(products)
        
    def users_for_ids(self, ids):
        return self.users[self.users['id'].isin(ids)]
    
    def products_for_indices(self, idxs):
        return self.products.iloc[idxs]
    
    def all_products(self):
        return self.products

Now we have to create the bound deployments in dependency order

In [51]:
bound_db_facade_deployment = DatabaseFacade.bind('/mnt/cluster_storage/ecom/users.ndjson', '/mnt/cluster_storage/ecom/cat_with_embeddings')

bound_rec_deployment = Recommender.bind(base_model_path, 3, 1000, 1000, bound_db_facade_deployment)

bound_ingress = Ingress.bind(bound_rec_deployment)

And run

In [52]:
app_handle = serve.run(bound_ingress)

(ProxyActor pid=62964) INFO 2026-08-24 14:45:45,876 proxy 100.125.138.24 -- Proxy starting on node f987380542f8cbfb6253e55ae77bf5ade2c020d1e77df80460d6d8da (HTTP port: 8000).
INFO 2026-08-24 14:45:45,942 serve 54397 -- Started Serve in namespace "serve".
(ProxyActor pid=62964) INFO 2026-08-24 14:45:45,939 proxy 100.125.138.24 -- Got updated endpoints: {}.
(ServeController pid=62891) INFO 2026-08-24 14:45:46,029 controller 62891 -- Deploying new version of Deployment(name='DatabaseFacade', app='default') (initial target replicas: 1).
(ServeController pid=62891) INFO 2026-08-24 14:45:46,030 controller 62891 -- Deploying new version of Deployment(name='Recommender', app='default') (initial target replicas: 1).
(ServeController pid=62891) INFO 2026-08-24 14:45:46,031 controller 62891 -- Deploying new version of Deployment(name='Ingress', app='default') (initial target replicas: 1).
(ProxyActor pid=62964) INFO 2026-08-24 14:45:46,035 proxy 100.125.138.24 -- Got updated endpoints: {Deploymen

In [53]:
response = requests.post("http://localhost:8000/", json={ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f"})

response.json()

(ServeReplica:default:Ingress pid=29897, ip=100.113.73.39) INFO 2026-08-24 14:45:50,225 default_Ingress cfrcjvbf 5536f4b2-3e56-45ad-8616-a4f11874640d -- Started <ray.serve._private.router.SharedRouterLongPollClient object at 0x7c67b09cbdd0>.


{'item_id': {'395': 'SHO-C70NS1L7I',
  '487': 'CAT-X8Y6OKF0V',
  '751': 'VEN-489ZNGGOE'},
 'name': {'395': 'LED Under-Cabinet Light Bar (Slim Profile) (Red)',
  '487': 'Rice (Long Grain) (Low Sugar) (32 oz)',
  '751': 'Wireless Earbuds - Extended Battery (64GB, Teal)'},
 'desc': {'395': 'A slim LED light bar to brighten counters and workspaces. Easy to wipe clean and store between uses.',
  '487': 'Long grain rice that cooks fluffy for bowls and sides. Packed for freshness with resealable packaging.',
  '751': 'Compact wireless earbuds with a stable fit for daily listening. Includes a quick-start guide and standard charging support.'},
 'price': {'395': 13.0, '487': 5.99, '751': 92.0}}

Finally, let's implement a semantic search using our product description embeddings. The search component will also be a Deployment.

In [54]:
@serve.deployment()
class SemanticSearch():
    async def __init__(self, model, db):
        self.model = SentenceTransformer(model)
        self.db = db
        self.all_products_embeddings = (await self.db.all_products.remote())['desc_emb']
        
    async def search(self, query, matches):
        similarities = self.model.similarity(self.model.encode(query), self.all_products_embeddings)
        top_matches = similarities.flatten().topk(50).indices
        products = await self.db.products_for_indices.remote(np.array(top_matches))
        # rerank
        top_name_distances = torch.tensor([textdistance.lcsstr.similarity(query, prodname) for prodname in list(products['name'])]).topk(matches).indices
        results = products.iloc[np.array(top_name_distances)]
        
        return results

(ServeReplica:default:Recommender pid=29895, ip=100.113.73.39) INFO 2026-08-24 14:45:50,312 default_Recommender yvs1jk5a 5536f4b2-3e56-45ad-8616-a4f11874640d -- Started <ray.serve._private.router.SharedRouterLongPollClient object at 0x765a63717a50>.
(ServeReplica:default:Recommender pid=29895, ip=100.113.73.39) INFO 2026-08-24 14:45:50,355 default_Recommender yvs1jk5a 5536f4b2-3e56-45ad-8616-a4f11874640d -- CALL recommend OK 57.9ms
(ServeReplica:default:DatabaseFacade pid=29896, ip=100.113.73.39) /home/ray/anaconda3/lib/python3.11/site-packages/ray/serve/_private/replica.py:3153: UserWarning: Calling sync method 'users_for_ids' directly on the asyncio loop. In a future version, sync methods will be run in a threadpool by default. Ensure your sync methods are thread safe or keep the existing behavior by making them `async def`. Opt into the new behavior by setting RAY_SERVE_RUN_SYNC_IN_THREADPOOL=1.
(ServeReplica:default:DatabaseFacade pid=29896, ip=100.113.73.39)   warnings.warn(
(Serv

In [55]:
cached_embedding_model = "/mnt/cluster_storage/ecom/hf_cache/models--google--embeddinggemma-300m/snapshots/57c266a740f537b4dc058e1b0cda161fd15afa75"

bound_search = SemanticSearch.bind(cached_embedding_model, bound_db_facade_deployment)

We'll re-write the Ingress Deployment to support routing: if the service is called with a query, we'll route to that Deployment.

In [56]:
@serve.deployment()
class Ingress:
    def __init__(self, recommender, search):
        self.recommender = recommender
        self.search = search

    async def __call__(self, request: Request):  # __call__ takes a Request object
        user = await request.json()
        recommendations = (await self.recommender.recommend.remote([user['id']]))[['item_id', 'name', 'desc', 'price']]
        result = { "recommendations" : recommendations.to_json() }
        if "query" in user:
            search_results = (await self.search.search.remote(user['query'], 5))[['item_id', 'name', 'desc', 'price']]
            result["search_results"] = search_results.to_json()
        
        return json.dumps(result)

In [57]:
bound_ingress = Ingress.bind(bound_rec_deployment, bound_search)

In [58]:
app_handle = serve.run(bound_ingress)

INFO 2026-08-24 14:45:51,198 serve 54397 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.
(ServeController pid=62891) INFO 2026-08-24 14:45:51,264 controller 62891 -- Deploying new version of Deployment(name='DatabaseFacade', app='default') (initial target replicas: 1).
(ServeController pid=62891) INFO 2026-08-24 14:45:51,264 controller 62891 -- Deploying new version of Deployment(name='Recommender', app='default') (initial target replicas: 1).
(ServeController pid=62891) INFO 2026-08-24 14:45:51,265 controller 62891 -- Deploying new version of Deployment(name='SemanticSearch', app='default') (initial target replicas: 1).
(ServeController pid=62891) INFO 2026-08-24 14:45:51,266 controller 62891 -- Deploying new version of Deployment(name='Ingress', app='default') (initial target replicas: 1).
(ServeController pid=62891) INFO 2026-08-24 14:45:51,373 controller 62891 -- Stopping 1 replicas of Deployment(name='DatabaseFacade', app='default') 

In [59]:
response = requests.post("http://localhost:8000/", json={ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f", "query" : "steel bols"})

response.json()

(ServeReplica:default:Ingress pid=30180, ip=100.113.73.39) INFO 2026-08-24 14:46:08,427 default_Ingress fy9pnoli 626a41a2-9cd4-485d-a967-75f1b7594b55 -- Started <ray.serve._private.router.SharedRouterLongPollClient object at 0x776eec275f10>.
(ServeReplica:default:DatabaseFacade pid=30177, ip=100.113.73.39) INFO 2026-08-24 14:46:08,542 default_DatabaseFacade jyeqhgt7 626a41a2-9cd4-485d-a967-75f1b7594b55 -- CALL users_for_ids OK 3.4ms
(ServeReplica:default:DatabaseFacade pid=30177, ip=100.113.73.39) INFO 2026-08-24 14:46:08,549 default_DatabaseFacade jyeqhgt7 626a41a2-9cd4-485d-a967-75f1b7594b55 -- CALL products_for_indices OK 2.4ms
(ServeReplica:default:Recommender pid=30178, ip=100.113.73.39) INFO 2026-08-24 14:46:08,516 default_Recommender 81l1uqbe 626a41a2-9cd4-485d-a967-75f1b7594b55 -- Started <ray.serve._private.router.SharedRouterLongPollClient object at 0x714a7994a490>.
(ServeReplica:default:Recommender pid=30178, ip=100.113.73.39) INFO 2026-08-24 14:46:08,553 default_Recommender

{'recommendations': '{"item_id":{"395":"SHO-C70NS1L7I","487":"CAT-X8Y6OKF0V","751":"VEN-489ZNGGOE"},"name":{"395":"LED Under-Cabinet Light Bar (Slim Profile) (Red)","487":"Rice (Long Grain) (Low Sugar) (32 oz)","751":"Wireless Earbuds - Extended Battery (64GB, Teal)"},"desc":{"395":"A slim LED light bar to brighten counters and workspaces. Easy to wipe clean and store between uses.","487":"Long grain rice that cooks fluffy for bowls and sides. Packed for freshness with resealable packaging.","751":"Compact wireless earbuds with a stable fit for daily listening. Includes a quick-start guide and standard charging support."},"price":{"395":13.0,"487":5.99,"751":92.0}}',
 'search_results': '{"item_id":{"212":"VEN-5CZU8AWJ0","103":"MKP-RDUTX467P","344":"HUB-AHNT2J7BI","83":"MKP-I3B7I0FK9","20":"MKP-AKVN0BJPT"},"name":{"212":"Stainless Steel Mixing Bowl Set (with Lid) (Teal)","103":"Stainless Steel Mixing Bowl Set (with Lid) (Black)","344":"Stainless Steel Mixing Bowl Set (Pink)","83":"Stain

In [60]:
response = requests.post("http://localhost:8000/", json={ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f" })

response.json()

(ServeReplica:default:DatabaseFacade pid=30177, ip=100.113.73.39) INFO 2026-08-24 14:46:09,306 default_DatabaseFacade jyeqhgt7 626a41a2-9cd4-485d-a967-75f1b7594b55 -- CALL products_for_indices OK 3.4ms
(ServeReplica:default:Ingress pid=30180, ip=100.113.73.39) INFO 2026-08-24 14:46:09,317 default_Ingress fy9pnoli 626a41a2-9cd4-485d-a967-75f1b7594b55 -- POST / 200 907.0ms
(ServeReplica:default:SemanticSearch pid=30179, ip=100.113.73.39) /home/ray/anaconda3/lib/python3.11/site-packages/sentence_transformers/util/tensor.py:30: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
(ServeReplica:default:SemanticSearch pid=30179, ip=100.113.73.39)   a = torch.tensor(a)
(ServeReplica:default:SemanticSearch pid=30179, ip=100.113.73.39) /tmp/ipykernel_54397/2868459157.py:11: DeprecationWa

(ServeReplica:default:DatabaseFacade pid=30177, ip=100.113.73.39) INFO 2026-08-24 14:46:10,206 default_DatabaseFacade jyeqhgt7 618b6ddc-e377-4ea0-8db0-51b6bf0f1761 -- CALL users_for_ids OK 2.7ms


{'recommendations': '{"item_id":{"395":"SHO-C70NS1L7I","487":"CAT-X8Y6OKF0V","751":"VEN-489ZNGGOE"},"name":{"395":"LED Under-Cabinet Light Bar (Slim Profile) (Red)","487":"Rice (Long Grain) (Low Sugar) (32 oz)","751":"Wireless Earbuds - Extended Battery (64GB, Teal)"},"desc":{"395":"A slim LED light bar to brighten counters and workspaces. Easy to wipe clean and store between uses.","487":"Long grain rice that cooks fluffy for bowls and sides. Packed for freshness with resealable packaging.","751":"Compact wireless earbuds with a stable fit for daily listening. Includes a quick-start guide and standard charging support."},"price":{"395":13.0,"487":5.99,"751":92.0}}'}

In [61]:
! serve shutdown -y

(ServeReplica:default:DatabaseFacade pid=30177, ip=100.113.73.39) INFO 2026-08-24 14:46:10,212 default_DatabaseFacade jyeqhgt7 618b6ddc-e377-4ea0-8db0-51b6bf0f1761 -- CALL products_for_indices OK 2.3ms
(ServeReplica:default:Recommender pid=30178, ip=100.113.73.39) INFO 2026-08-24 14:46:10,215 default_Recommender 81l1uqbe 618b6ddc-e377-4ea0-8db0-51b6bf0f1761 -- CALL recommend OK 13.8ms
(ServeReplica:default:Ingress pid=30180, ip=100.113.73.39) INFO 2026-08-24 14:46:10,219 default_Ingress fy9pnoli 618b6ddc-e377-4ea0-8db0-51b6bf0f1761 -- POST / 200 23.5ms
(ServeController pid=62891) INFO 2026-08-24 14:46:12,518 controller 62891 -- Removing 1 replica from Deployment(name='DatabaseFacade', app='default').
(ServeController pid=62891) INFO 2026-08-24 14:46:12,518 controller 62891 -- Removing 1 replica from Deployment(name='Recommender', app='default').
(ServeController pid=62891) INFO 2026-08-24 14:46:12,518 controller 62891 -- Removing 1 replica from Deployment(name='Ingress', app='default')

(raylet) Task ServeController.graceful_shutdown failed. There are infinite retries remaining, so the task will be retried. Error: The actor is dead because it was killed by `ray.kill`.


(ServeController pid=62891) INFO 2026-08-24 14:46:14,555 controller 62891 -- Replica(id='jyeqhgt7', deployment='DatabaseFacade', app='default') is stopped.
(ServeController pid=62891) INFO 2026-08-24 14:46:14,556 controller 62891 -- Replica(id='81l1uqbe', deployment='Recommender', app='default') is stopped.
(ServeController pid=62891) INFO 2026-08-24 14:46:14,557 controller 62891 -- Replica(id='fy9pnoli', deployment='Ingress', app='default') is stopped.
(ServeController pid=62891) INFO 2026-08-24 14:46:14,557 controller 62891 -- Replica(id='hi50v08g', deployment='SemanticSearch', app='default') is stopped.


2026-08-24 14:46:15,225	SUCC scripts.py:789 -- Sent shutdown request; applications will be deleted asynchronously.


## Deploying via the CLI

Inspect the refactored code in 
* `recommend.py`
* `search_and_recommend.py`

Pay extra attention to...
* file paths that will be available/visible at runtime
* imported code (and working dir)
* environment vars and other requirements
* ambiguous names (scoping and namespacing isn't perfect since Python is not yet designed for a distributed runtime)

In [62]:
! serve build search_and_recommend:bound_ingress -o serve_config.yaml

2026-08-24 14:46:24,568	INFO scripts.py:958 -- The auto-generated application names default to `app1`, `app2`, ... etc. Rename as necessary.



In [63]:
! cat serve_config.yaml

# This file was generated using the `serve build` command on Ray v2.55.1.

proxy_location: EveryNode

http_options:
  host: 0.0.0.0
  port: 8000

grpc_options:
  port: 9000
  grpc_servicer_functions: []

logging_config:
  encoding: TEXT
  log_level: INFO
  logs_dir: null
  enable_access_log: true
  additional_log_standard_attrs: []

applications:
- name: app1
  route_prefix: /
  import_path: search_and_recommend:bound_ingress
  runtime_env: {}
  deployments:
  - name: DatabaseFacade
  - name: Recommender
  - name: SemanticSearch
  - name: Ingress


We'll update this to produce `serve_updated.yaml`

(reference https://docs.ray.io/en/latest/serve/production-guide/config.html)

```yaml
# This file was generated using the `serve build` command on Ray v2.55.1.

proxy_location: EveryNode

http_options:
  host: 0.0.0.0
  port: 8000

grpc_options:
  port: 9000
  grpc_servicer_functions: []

logging_config:
  encoding: JSON
  log_level: INFO
  logs_dir: null
  enable_access_log: true
  additional_log_standard_attrs: []

applications:
- name: search_recommend
  route_prefix: /
  import_path: search_and_recommend:bound_ingress
  runtime_env:
    working_dir: "https://anyscale-public-materials-use2.s3.us-east-2.amazonaws.com/recommend.zip"
  deployments:
  - name: DatabaseFacade
    num_replicas: 4
    graceful_shutdown_wait_loop_s: 2.0
    graceful_shutdown_timeout_s: 20.0
    health_check_period_s: 10.0
    health_check_timeout_s: 30.0
        
  - name: Recommender
    num_replicas: 2
    max_ongoing_requests: 100
    ray_actor_options:
      num_cpus: 1
      num_gpus: 0.5
      
  - name: SemanticSearch
    autoscaling_config:
      min_replicas: 2
      max_replicas: 4

  - name: Ingress
    autoscaling_config:
      min_replicas: 2
      max_replicas: 4
```

We'll deploy this service from the CLI

In [64]:
! serve deploy serve_updated.yaml

2026-08-24 14:46:29,467	INFO scripts.py:247 -- Deploying from config file: 'serve_updated.yaml'.
2026-08-24 14:46:33,532	SUCC scripts.py:367 -- 
Sent deploy request successfully.
 * Use `serve status` to check applications' statuses.
 * Use `serve config` to see the current application config(s).



(ProxyActor pid=63618) INFO 2026-08-24 14:46:33,450 proxy 100.125.138.24 -- Proxy starting on node f987380542f8cbfb6253e55ae77bf5ade2c020d1e77df80460d6d8da (HTTP port: 8000).
(ServeController pid=63564) INFO 2026-08-24 14:46:33,522 controller 63564 -- Deploying new app 'search_recommend'.
(ServeController pid=63564) INFO 2026-08-24 14:46:33,523 controller 63564 -- Importing and building app 'search_recommend'.
(ProxyActor pid=63618) INFO 2026-08-24 14:46:33,514 proxy 100.125.138.24 -- Got updated endpoints: {}.


In [65]:
response = requests.post("http://localhost:8000/", json='{ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f", "query" : "set of bowls"}')

response.json()

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

(build_serve_application pid=63693) INFO 2026-08-24 14:46:36,278 controller build_search_recommend_63693 -- Importing application 'search_recommend'.
(ServeController pid=63564) INFO 2026-08-24 14:46:41,588 controller 63564 -- Imported and built app 'search_recommend' successfully.
(ServeController pid=63564) INFO 2026-08-24 14:46:41,647 controller 63564 -- Deploying new version of Deployment(name='DatabaseFacade', app='search_recommend') (initial target replicas: 4).
(ServeController pid=63564) INFO 2026-08-24 14:46:41,648 controller 63564 -- Deploying new version of Deployment(name='Recommender', app='search_recommend') (initial target replicas: 2).
(ServeController pid=63564) INFO 2026-08-24 14:46:41,650 controller 63564 -- Registering autoscaling state for deployment Deployment(name='SemanticSearch', app='search_recommend')
(ServeController pid=63564) INFO 2026-08-24 14:46:41,650 controller 63564 -- Deploying new version of Deployment(name='SemanticSearch', app='search_recommend') 

(autoscaler +10m39s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.


(TrainController pid=68185) Requesting resources: {'CPU': 1} * 2
(TrainController pid=68185) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=68185) Attempting to start training worker group of size 2 with the following resources: [{'CPU': 1}] * 2


(RayTrainWorker pid=31296, ip=100.93.121.85) [Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


...

(RayTrainWorker pid=32803, ip=100.93.121.85) [Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1


(RayTrainWorker pid=32803, ip=100.93.121.85) Reporting training result 4: TrainingReport(checkpoint=None, metrics={'loss': 5.6156110763549805}, validation=False)
(RayTrainWorker pid=32802, ip=100.93.121.85) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-37.319094)
(RayTrainWorker pid=32802, ip=100.93.121.85) Reporting training result 4: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-37.319094), metrics={'loss': 5.590732574462891}, validation=False)
(SplitCoordinator pid=70340) Registered dataset logger for dataset train_50_4
(SplitCoordinator pid=70340) Starting execution of Dataset train_50_4. Full logs are in /tmp/ray/session_2026-08-24_11-19-54_832609_2820/logs/ray-data
(SplitCoordinator pid=70340) Execution plan of Dataset train_50_4: InputDataBuffer[Input] -> OutputSplitter[split(2,

(pid=70340) Running Dataset train_50_4.: 0.00 row [00:00, ? row/s]

(pid=70340) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(RayTrainWorker pid=32803, ip=100.93.121.85) Reporting training result 5: TrainingReport(checkpoint=None, metrics={'loss': 5.613339900970459}, validation=False)
(RayTrainWorker pid=32802, ip=100.93.121.85) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-39.415962)
(RayTrainWorker pid=32802, ip=100.93.121.85) Reporting training result 5: TrainingReport(checkpoint=Checkpoint(filesystem=local, path=/mnt/cluster_storage/ray_train_run-2026-08-24_14-56-22/checkpoint_2026-08-24_14-56-39.415962), metrics={'loss': 5.588497161865234}, validation=False)
(TrainController pid=70048) [State Transition] RUNNING -> SHUTTING_DOWN.
(TrainController pid=70048) [State Transition] SHUTTING_DOWN -> FINISHED.


In [ ]:
! serve shutdown -y